# Music Recommendation Modeling

Offline candidate generation, ALS personalization, item-to-item similarities, ranking, and service artifacts for a music recommendation system.

Outputs were stripped for public release. Re-run the notebook after configuring the required private data access or local artifact paths.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.sparse as sp
from sklearn.preprocessing import LabelEncoder
from implicit.als import AlternatingLeastSquares
%matplotlib inline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


In [ ]:
tracks = pd.read_parquet('tracks.parquet')
catalog = pd.read_parquet('catalog_names.parquet')
interactions = pd.read_parquet('interactions.parquet')


In [ ]:
print('tracks shape', tracks.shape)
print('catalog shape', catalog.shape)
print('interactions shape', interactions.shape)


In [ ]:
print('\ntracks dtypes')
print(tracks.dtypes)

print('\ncatalog dtypes')
print(catalog.dtypes)

print('\ninteractions dtypes')
print(interactions.dtypes)


In [ ]:
print('\ntracks sample')
display(tracks.sample(5, random_state=52))

print('\ncatalog sample')
display(catalog.sample(5, random_state=52))

print('\ninteractions sample')
display(interactions.sample(5, random_state=52))


In [ ]:
# See the project README for context.
def list_len(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x)
    return 0

empty_stats = pd.DataFrame({
    'artists_empty_share': tracks['artists'].apply(list_len).eq(0).mean(),
    'albums_empty_share':  tracks['albums'].apply(list_len).eq(0).mean(),
    'genres_empty_share':  tracks['genres'].apply(list_len).eq(0).mean(),
}, index=[0])
print('\nshare of empty lists in tracks')
display(empty_stats)


In [ ]:
# See the project README for context.
catalog = catalog.copy()
catalog['id_num'] = pd.to_numeric(catalog['id'], errors='coerce')
known_tracks  = set(catalog.loc[catalog['type'] == 'track',  'id_num'].dropna().astype('int64'))
known_artists = set(catalog.loc[catalog['type'] == 'artist', 'id_num'].dropna().astype('int64'))
known_albums  = set(catalog.loc[catalog['type'] == 'album',  'id_num'].dropna().astype('int64'))
known_genres  = set(catalog.loc[catalog['type'] == 'genre',  'id_num'].dropna().astype('int64'))

# See the project README for context.
artist_ids = pd.to_numeric(tracks['artists'].explode(), errors='coerce').dropna().astype('int64')
album_ids  = pd.to_numeric(tracks['albums'].explode(),  errors='coerce').dropna().astype('int64')
genre_ids  = pd.to_numeric(tracks['genres'].explode(),  errors='coerce').dropna().astype('int64')

unknown_counts = pd.Series({
    'unknown_track_ids_in_names': len(set(tracks['track_id']) - known_tracks),
    'unknown_artist_ids': int((~artist_ids.isin(known_artists)).sum()),
    'unknown_album_ids':  int((~album_ids.isin(known_albums)).sum()),
    'unknown_genre_ids':  int((~genre_ids.isin(known_genres)).sum()),
})
print('\nunknown id counts')
display(unknown_counts.to_frame('count'))


In [ ]:
# See the project README for context.
started_at_dt = pd.to_datetime(interactions['started_at'], errors='coerce', utc=True)
print('\nstarted_at null share', started_at_dt.isna().mean())
print('started_at min', started_at_dt.min())
print('started_at max', started_at_dt.max())

dups = interactions.duplicated(subset=['user_id', 'track_id', 'track_seq']).sum()
print('duplicates by user track track_seq', dups)

print('\nunique users', interactions['user_id'].nunique())
print('unique tracks in interactions', interactions['track_id'].nunique())


In [ ]:
# See the project README for context.

tracks = tracks.copy()

def to_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)
    if x is None or x is pd.NA:
        return []
    try:
        return [] if pd.isna(x) else [int(x)]
    except Exception:
        return [int(x)]

for col in ['albums', 'artists', 'genres']:
    tracks[col] = tracks[col].map(to_list)
    tracks[col] = tracks[col].map(lambda xs: [int(v) for v in xs if pd.notna(v)])

# See the project README for context.
known_genres = set(catalog.loc[catalog['type'] == 'genre', 'id'].astype('int64'))

# See the project README for context.
before = int(tracks['genres'].explode().shape[0])
tracks['genres'] = tracks['genres'].map(lambda xs: [g for g in xs if g in known_genres])
after = int(tracks['genres'].explode().shape[0])
print('removed_unknown_genre_ids', before - after)

# See the project README for context.
empty_stats = pd.DataFrame({
    'artists_empty_share': tracks['artists'].map(len).eq(0).mean(),
    'albums_empty_share':  tracks['albums'].map(len).eq(0).mean(),
    'genres_empty_share':  tracks['genres'].map(len).eq(0).mean(),
}, index=[0])
display(empty_stats)

# See the project README for context.
uniq_inter_tracks = interactions['track_id'].drop_duplicates()
missing = (~uniq_inter_tracks.isin(tracks['track_id'])).sum()
print('unique_interaction_track_ids_missing_in_tracks', int(missing))


In [ ]:
tracks['track_id'] = tracks['track_id'].astype('int32')
interactions['track_id'] = interactions['track_id'].astype('int32')
interactions['user_id'] = interactions['user_id'].astype('int32')

print('tracks id unique', tracks['track_id'].is_unique)

# See the project README for context.
dup_ut = interactions.duplicated(subset=['user_id', 'track_id']).sum()
print('duplicates user track', int(dup_ut))

# See the project README for context.
print('date range', interactions['started_at'].min(), interactions['started_at'].max())

# See the project README for context.
print('empty shares', {
    'artists': float(tracks['artists'].map(len).eq(0).mean()),
    'albums': float(tracks['albums'].map(len).eq(0).mean()),
    'genres': float(tracks['genres'].map(len).eq(0).mean()),
})


In [ ]:
user_tracks = interactions.groupby('user_id').size().rename('n_tracks')

print('users', user_tracks.shape[0])
print('mean', round(user_tracks.mean(), 2))
print('median', int(user_tracks.median()))
print('p90', int(user_tracks.quantile(0.90)))
print('p99', int(user_tracks.quantile(0.99)))

to_plot = user_tracks.clip(upper=user_tracks.quantile(0.99))

plt.figure(figsize=(8, 4))
to_plot.hist(bins=100)
# Plot title removed during public cleanup.
plt.xlabel(' ')
plt.ylabel(' ')
plt.show()


In [ ]:
top_tracks = (
    interactions.groupby('track_id')
    .agg(users=('user_id', 'nunique'), events=('user_id', 'count'))
    .reset_index()
    .sort_values(['users', 'events'], ascending=False)
)

track_names = catalog.loc[catalog['type'] == 'track', ['id', 'name']] \
                     .rename(columns={'id': 'track_id', 'name': 'track_name'})

top_tracks_named = top_tracks.merge(track_names, on='track_id', how='left')

display(top_tracks_named.head(20))


In [ ]:
# See the project README for context.
track_genres = tracks[['track_id', 'genres']].explode('genres').dropna()
track_genres['genres'] = track_genres['genres'].astype('int64')

# See the project README for context.
events_per_track = interactions.groupby('track_id').size().rename('events').reset_index()

# See the project README for context.
genre_pop = (
    track_genres.merge(events_per_track, on='track_id', how='left')
    .fillna({'events': 0})
    .groupby('genres', as_index=False)
    .agg(events=('events', 'sum'), tracks=('track_id', 'nunique'))
)

# See the project README for context.
genre_names = catalog.loc[catalog['type'] == 'genre', ['id', 'name']] \
                     .rename(columns={'id': 'genres', 'name': 'genre_name'})
genre_pop = genre_pop.merge(genre_names, on='genres', how='left')

genre_pop = genre_pop.sort_values('events', ascending=False)

display(genre_pop.head(30))


In [ ]:
all_tracks = pd.Index(tracks['track_id'].unique())
inter_tracks = pd.Index(interactions['track_id'].unique())
not_listened = all_tracks.difference(inter_tracks)

print('tracks total', len(all_tracks))
print('tracks in interactions', len(inter_tracks))
print('tracks with zero listens', len(not_listened))

if len(not_listened) > 0:
    sample = pd.DataFrame({'track_id': not_listened[:20]})
    names = catalog.loc[catalog['type'] == 'track', ['id', 'name']].rename(columns={'id': 'track_id', 'name': 'track_name'})
    display(sample.merge(names, on='track_id', how='left'))


In [ ]:
# See the project README for context.
items = tracks.rename(columns={'track_id': 'item_id'}).copy()
events = interactions.rename(columns={'track_id': 'item_id'}).copy()

# See the project README for context.
items['item_id'] = items['item_id'].astype('int32')
events['item_id'] = events['item_id'].astype('int32')
events['user_id'] = events['user_id'].astype('int32')

print('items shape', items.shape)
print('events shape', events.shape)
print('items columns', list(items.columns))
print('events columns', list(events.columns))


In [ ]:
# os.environ['AWS_ACCESS_KEY_ID'] = '...'
# os.environ['AWS_SECRET_ACCESS_KEY'] = '...'
# os.environ['AWS_DEFAULT_REGION'] = '...'
# os.environ['AWS_S3_ENDPOINT'] = 'https://storage.yandexcloud.net'

# bucket = '...'
# prefix = 'recsys/data'

# items_path = f's3://{bucket}/{prefix}/items.parquet'
# events_path = f's3://{bucket}/{prefix}/events.parquet'

# items.to_parquet(items_path, index=False, storage_options={
#     'key': os.environ['AWS_ACCESS_KEY_ID'],
#     'secret': os.environ['AWS_SECRET_ACCESS_KEY'],
#     'client_kwargs': {'endpoint_url': os.environ['AWS_S3_ENDPOINT']}
# })

# events.to_parquet(events_path, index=False, storage_options={
#     'key': os.environ['AWS_ACCESS_KEY_ID'],
#     'secret': os.environ['AWS_SECRET_ACCESS_KEY'],
#     'client_kwargs': {'endpoint_url': os.environ['AWS_S3_ENDPOINT']}
# })

# print('saved items to', items_path)
# print('saved events to', events_path)


In [ ]:
# See the project README for context.


In [ ]:
items = pd.read_parquet('recsys/data/items.parquet')
events = pd.read_parquet('recsys/data/events.parquet')


In [ ]:
events['started_at'] = pd.to_datetime(events['started_at'])

split_date = pd.Timestamp('2022-12-16')

events_train = events.loc[events['started_at'] < split_date].copy()
events_test  = events.loc[events['started_at'] >= split_date].copy()

print('events train', events_train.shape)
print('events test', events_test.shape)
print('users train', events_train['user_id'].nunique())
print('users test', events_test['user_id'].nunique())


In [ ]:
common_users = set(events_train['user_id'].unique()) & set(events_test['user_id'].unique())
print('common users', len(common_users))


In [ ]:
pop = (
    events_train.groupby('item_id')
    .agg(users=('user_id', 'nunique'), events=('user_id', 'count'))
    .reset_index()
)
pop['score'] = pop['users']
pop = pop.sort_values('score', ascending=False)
pop['rank'] = np.arange(1, len(pop) + 1)


In [ ]:
os.makedirs('recsys/recommendations', exist_ok=True)
top_pop_path = 'recsys/recommendations/top_popular.parquet'
pop[['item_id', 'score', 'rank', 'users', 'events']].to_parquet(top_pop_path, index=False)
# Artifacts can also be stored in object storage.
print('saved', top_pop_path, 'rows', len(pop))
display(pop.head(10))


In [ ]:
# See the project README for context.
user_le = LabelEncoder()
item_le = LabelEncoder()
user_le.fit(events_train['user_id'])
item_le.fit(items['item_id'])

# See the project README for context.
u = user_le.transform(events_train['user_id'].values)
i = item_le.transform(events_train['item_id'].values)
data = np.ones(len(u), dtype=np.float32)

n_users = len(user_le.classes_)
n_items = len(item_le.classes_)
ui = sp.csr_matrix((data, (u, i)), shape=(n_users, n_items))
iu = ui.T.tocsr()

print('matrix shape', ui.shape, 'nnz', ui.nnz)


In [ ]:
from threadpoolctl import threadpool_limits
threadpool_limits(1, "blas")


In [ ]:
als = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=15, random_state=0)
als.fit(ui)


In [ ]:
N = 15
BATCH = 20000
users_common = np.array(sorted(common_users), dtype=np.int64)
users_common_enc = user_le.transform(users_common)

parts = []
for s in range(0, len(users_common_enc), BATCH):
    u_slice = users_common_enc[s:s+BATCH]
    I, S = als.recommend(u_slice, ui[u_slice], N=N, filter_already_liked_items=True)
    df = pd.DataFrame({
        'user_id_enc': np.repeat(u_slice, N),
        'item_id_enc': I.reshape(-1),
        'score': S.reshape(-1).astype(np.float32)
    })
    parts.append(df)
    print('done users', s + len(u_slice))

recs_df = pd.concat(parts, ignore_index=True)
recs_df['user_id'] = user_le.inverse_transform(recs_df['user_id_enc'])
recs_df['item_id'] = item_le.inverse_transform(recs_df['item_id_enc'])
recs_df = recs_df[['user_id', 'item_id', 'score']]
# Artifacts can also be stored in object storage.
personal_path = 'recsys/recommendations/personal_als.parquet'
recs_df.to_parquet(personal_path, index=False)
print('saved', personal_path, 'users', recs_df['user_id'].nunique(), 'rows', len(recs_df))
display(recs_df.head())


In [ ]:
# See the project README for context.

MIN_CNT = 50
RECENT_FROM = pd.Timestamp('2022-11-01')
K = 10
BATCH = 5000

item_cnt = events_train.groupby('item_id').size()
popular = item_cnt[item_cnt >= MIN_CNT].index
recent = events_train.loc[events_train['started_at'] >= RECENT_FROM, 'item_id'].unique()
cand = np.array(sorted(set(popular) | set(recent)), dtype=np.int64)
cand_enc = item_le.transform(cand)

print('candidates', len(cand_enc))

parts = []
with threadpool_limits(1, 'blas'):
    for s in range(0, len(cand_enc), BATCH):
        ids = cand_enc[s:s+BATCH]
        sim_ids, sim_scores = als.similar_items(ids, N=K+1)
        df = pd.DataFrame({
            'item_id_enc': np.repeat(ids, K+1),
            'sim_item_id_enc': sim_ids.reshape(-1),
            'score': sim_scores.reshape(-1).astype(np.float32)
        })
        df = df[df['item_id_enc'] != df['sim_item_id_enc']].copy()
        parts.append(df)
        print('done', s + len(ids))

sim_df = pd.concat(parts, ignore_index=True)
sim_df['item_id_1'] = item_le.inverse_transform(sim_df['item_id_enc'])
sim_df['item_id_2'] = item_le.inverse_transform(sim_df['sim_item_id_enc'])
sim_df = sim_df[['item_id_1', 'item_id_2', 'score']]
# Artifacts can also be stored in object storage.
sim_df.to_parquet('recsys/recommendations/similar.parquet', index=False)
print('saved recsys/recommendations/similar.parquet rows', len(sim_df))


In [ ]:
recs_df = pd.read_parquet('recsys/recommendations/personal_als.parquet')


In [ ]:

# See the project README for context.
label_split = pd.Timestamp('2022-12-24')

events_labels  = events_test.loc[events_test['started_at'] <  label_split, ['user_id','item_id']].drop_duplicates()
events_holdout = events_test.loc[events_test['started_at'] >= label_split, ['user_id','item_id']].drop_duplicates()

# See the project README for context.
pop = pd.read_parquet('recsys/recommendations/top_popular.parquet')[['item_id','users','events']]
pop = pop.rename(columns={'users':'pop_users','events':'pop_events'})

item_last_ts = events_train.groupby('item_id')['started_at'].max()
item_recency_days = (label_split - item_last_ts).dt.days.rename('recency_days').to_frame().reset_index()

item_feats = pop.merge(item_recency_days, on='item_id', how='outer')
item_feats[['pop_users','pop_events','recency_days']] = item_feats[['pop_users','pop_events','recency_days']].fillna(0)

# See the project README for context.
als_cands = recs_df.copy()

# See the project README for context.
label_users = np.intersect1d(events_labels['user_id'].unique(), als_cands['user_id'].unique())
train_cands = als_cands[als_cands['user_id'].isin(label_users)].copy()

# See the project README for context.
events_labels['target'] = 1
train_cands = train_cands.merge(events_labels, on=['user_id','item_id'], how='left')
train_cands['target'] = train_cands['target'].fillna(0).astype('int8')


In [ ]:
#Add item features
train_cands = train_cands.merge(item_feats, on='item_id', how='left')
train_cands[['pop_users','pop_events','recency_days']] = train_cands[['pop_users','pop_events','recency_days']].fillna(0)

# See the project README for context.
pos_users = train_cands.groupby('user_id')['target'].sum()
good_users = pos_users[pos_users > 0].index
train_cands = train_cands[train_cands['user_id'].isin(good_users)]

# See the project README for context.
neg_k = 4
train_pos = train_cands[train_cands['target'] == 1]
train_neg = (
    train_cands[train_cands['target'] == 0]
    .groupby('user_id', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), neg_k), random_state=0))
)
train_data = pd.concat([train_pos, train_neg], ignore_index=True)

print('train users', train_data['user_id'].nunique(), 'rows', len(train_data))
train_data.head()


In [ ]:
# Train and rank candidates

from catboost import CatBoostClassifier, Pool

features = ['score','pop_users','pop_events','recency_days']
target = 'target'

cb = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    random_seed=0,
    verbose=100
)

pool_train = Pool(train_data[features], label=train_data[target])
cb.fit(pool_train)


In [ ]:
# See the project README for context.
holdout_users = np.intersect1d(events_holdout['user_id'].unique(), als_cands['user_id'].unique())
rank_cands = als_cands[als_cands['user_id'].isin(holdout_users)].copy()
rank_cands = rank_cands.merge(item_feats, on='item_id', how='left')
rank_cands[['pop_users','pop_events','recency_days']] = rank_cands[['pop_users','pop_events','recency_days']].fillna(0)

pred = cb.predict_proba(Pool(rank_cands[features]))[:, 1]
rank_cands['cb_score'] = pred

rank_cands = rank_cands.sort_values(['user_id','cb_score'], ascending=[True, False])
rank_cands['rank'] = rank_cands.groupby('user_id').cumcount() + 1

max_k = 100
final_recs = rank_cands[rank_cands['rank'] <= max_k][['user_id','item_id','cb_score','rank']].copy()

out_path = 'recsys/recommendations/recommendations.parquet'
# Artifacts can also be stored in object storage.
final_recs.to_parquet(out_path, index=False)
print('saved', out_path, 'users', final_recs['user_id'].nunique(), 'rows', len(final_recs))

final_recs.head(10)


In [ ]:
TOPK = 5
COV_K = 100
EVAL_N = 100_000
NOVELTY_N = 20_000

# See the project README for context.
pop = pd.read_parquet('recsys/recommendations/top_popular.parquet')[['item_id','rank']]

# See the project README for context.
users_gt = events_holdout['user_id'].unique()
users_als = recs_df['user_id'].unique()
users_final = final_recs['user_id'].unique()
eval_users = np.intersect1d(users_gt, np.intersect1d(users_als, users_final))

if len(eval_users) > EVAL_N:
    rng = np.random.default_rng(0)
    eval_users = np.sort(rng.choice(eval_users, size=EVAL_N, replace=False))

# ground truth
gt_small = events_holdout.loc[events_holdout['user_id'].isin(eval_users), ['user_id','item_id']].drop_duplicates()

# See the project README for context.
als_small = (
    recs_df[recs_df['user_id'].isin(eval_users)]
    .sort_values(['user_id','score'], ascending=[True, False])
    .groupby('user_id').head(TOPK)[['user_id','item_id']]
)
final_small = (
    final_recs[final_recs['user_id'].isin(eval_users)]
    .sort_values(['user_id','cb_score'], ascending=[True, False])
    .groupby('user_id').head(TOPK)[['user_id','item_id']]
)


In [ ]:
top_items_k = pop.sort_values('rank').head(TOPK)['item_id'].tolist()
top_small = pd.DataFrame({
    'user_id': np.repeat(eval_users, TOPK),
    'item_id': np.tile(top_items_k, len(eval_users))
})

def pr_at_k(pred_df, gt_df, users, k):
    hits = pred_df.merge(gt_df, on=['user_id','item_id'], how='inner').groupby('user_id').size()
    hits = hits.reindex(users, fill_value=0)
    prec = float((hits / k).mean())
    gt_counts = gt_df.groupby('user_id').size().reindex(users, fill_value=0)
    mask = gt_counts > 0
    rec = float((hits[mask] / gt_counts[mask]).mean())
    return prec, rec

prec_pop,  rec_pop  = pr_at_k(top_small,   gt_small, eval_users, TOPK)
prec_als,  rec_als  = pr_at_k(als_small,   gt_small, eval_users, TOPK)
prec_final,rec_final= pr_at_k(final_small, gt_small, eval_users, TOPK)


In [ ]:
# coverage
catalog_size = items['item_id'].nunique()
top100 = pop.sort_values('rank').head(COV_K)['item_id'].tolist()
cov_pop = float(len(set(top100)) / catalog_size)

als100_nuniq = (
    recs_df[recs_df['user_id'].isin(eval_users)]
    .sort_values(['user_id','score'], ascending=[True, False])
    .groupby('user_id').head(COV_K)['item_id'].nunique()
)
cov_als = float(als100_nuniq / catalog_size)

final100_nuniq = (
    final_recs[final_recs['user_id'].isin(eval_users)]
    .sort_values(['user_id','cb_score'], ascending=[True, False])
    .groupby('user_id').head(COV_K)['item_id'].nunique()
)
cov_final = float(final100_nuniq / catalog_size)


In [ ]:
# See the project README for context.
if len(eval_users) > NOVELTY_N:
    rng = np.random.default_rng(1)
    nov_users = np.sort(rng.choice(eval_users, size=NOVELTY_N, replace=False))
else:
    nov_users = eval_users

def novelty_at_k(pred_df, users, k):
    pred_k = (
        pred_df[pred_df['user_id'].isin(users)]
        .groupby('user_id').head(k)
    )
    train_seen = (
        events_train.loc[events_train['user_id'].isin(users), ['user_id','item_id']]
        .drop_duplicates()
    )
    seen_cnt = (
        pred_k.merge(train_seen, on=['user_id','item_id'], how='inner')
        .groupby('user_id').size()
        .reindex(users, fill_value=0)
    )
    denom = (
        pred_k.groupby('user_id').size()
        .reindex(users, fill_value=0)
        .replace(0, np.nan)
    )
    nov = 1 - (seen_cnt / denom)
    return float(nov.mean(skipna=True))

nov_pop   = novelty_at_k(top_small,   nov_users, TOPK)
nov_als   = novelty_at_k(als_small,   nov_users, TOPK)
nov_final = novelty_at_k(final_small, nov_users, TOPK)

metrics = pd.DataFrame({
    'recs_type':    ['top_popular','personal_als','final_ranked'],
    'precision@5':  [prec_pop, prec_als, prec_final],
    'recall@5':     [rec_pop,  rec_als,  rec_final],
    'coverage@100': [cov_pop,  cov_als,  cov_final],
    'novelty@5':    [nov_pop,  nov_als,  nov_final],
})

metrics


In [ ]:
metrics_path = 'recsys/recommendations/metrics.parquet'
# Artifacts can also be stored in object storage.
metrics.to_parquet(metrics_path, index=False)
print('saved', metrics_path)


In [ ]:
cb.save_model('recsys/recommendations/catboost_ranker.cbm')
print('saved recsys/recommendations/catboost_ranker.cbm')
